# L10d Lab: Evaluation of Bandit Problem Algorithms
In this lab, we'll continue our discussion [of multi-armed bandit problems](https://en.wikipedia.org/wiki/Multi-armed_bandit), where the objective of a bandit algorithm is to discover the action (or "arm") that provides the highest expected reward. We look at the performance of the explore-first, $\epsilon$-greedy, and UCB1 algorithms on a simple stochastic bandit problem, then implement Thompson sampling and use it to solve a binary Bernoulli bandit problem.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * __Run three bandit algorithms on the same problem:__ The explore-first, epsilon-greedy, and UCB1 algorithms each use the rewards seen so far to decide which arm to pull next. We'll run all three on a seven-arm bandit and count how often each one pulls each arm.
> * __Compare regret across repeated games:__ Regret measures how much reward an algorithm gives up compared with always pulling the best arm. We'll compute the regret of one game, then average many games and compare how fast the average grows with the regret bounds from the lecture.
> * __Implement Thompson sampling:__ Thompson sampling keeps a Beta distribution for the success probability of each arm and pulls the arm whose sample from that distribution is highest. We'll complete a Thompson sampling function and check that it learns which arm is best.

In [the lecture](../L10c/CHEME-5800-L10c-Lecture-StochasticMultiarmBandits-Fall-2026.ipynb), we introduced the stochastic multi-armed bandit problem, regret, and algorithms that balance exploration and exploitation. Here, we run these algorithms on bandits where we know the answer, so we can see how each one finds the best arm.

Let's get started!
___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this lab's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [ ]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"));

# Choose one theme; after changing it, rerun this cell and the plotting cells -
theme(:default)   # light background
# theme(:dark)   # dark background

In addition to standard Julia libraries, we'll use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

Your Thompson sampling function goes in [src/Compute.jl](src/Compute.jl). We call [the `L10dThompson.my_thompson_sampling(...)` function](src/Compute.jl) with its module name, `L10dThompson`, in front, so that re-running the setup cell picks up your edits.

### The Arms
First, let's implement the `world(...)` function. This function takes the action $a$ from our agent and returns the reward associated with choosing this action, i.e., $r\sim\mathcal{D}_{a}$. We'll use [a Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution) to model the rewards, so every reward is a number between 0 and 1. A Beta distribution has two parameters $(\alpha,\beta)$ which determine its shape, and its expectation is $\alpha/(\alpha+\beta)$.

We store the reward distribution of each arm in the `arms::Array{Beta{Float64},1}` array; the bandit algorithms see only the rewards:

In [ ]:
arms = [
    Beta(2.0, 2.0),   # arm 1
    Beta(10.0, 2.0),  # arm 2
    Beta(2.0, 10.0),  # arm 3
    Beta(6.0, 20.0),  # arm 4
    Beta(20.0, 6.0),  # arm 5
    Beta(20.0, 10.0), # arm 6
    Beta(10.0, 20.0)  # arm 7
]; # the reward distribution 𝒟ₐ of each arm

world = a -> rand(arms[a]); # a -> ... defines a function of a: world(a) draws one independent reward from the fixed distribution of arm a

Which arm has the highest expected reward? Only the world knows this answer, the bandit algorithms do not! Let's tabulate the parameters, mean reward, and standard deviation of each arm:

In [ ]:
let

    # initialize -
    df = DataFrame(); # one row per arm

    # compute the mean and the standard deviation of the reward of each arm -
    for a ∈ eachindex(arms)
        d = arms[a]; # the reward distribution of arm a
        (α, β) = params(d); # the two parameters of the Beta distribution
        push!(df, (action = a, alpha = α, beta = β,
            mean_reward = round(mean(d), digits = 3), sd_reward = round(std(d), digits = 3)));
    end

    # show the table -
    pretty_table(df;
        show_first_column_label_only = true,
        display_size = (-1, -1), # show every row and column
        alignment = :r # right-align every column
    );
end

__What do we see?__ The winner is the arm with the highest expected reward, which is set by the parameters of its Beta distribution: arm 2, with a mean reward of $10/12\approx 0.833$.

Let's plot the reward distributions of arm 2 and another arm, arm 6. [The `pdf(...)` function](https://juliastats.org/Distributions.jl/stable/univariate/#Distributions.pdf-Tuple{UnivariateDistribution,%20Real}) evaluates the probability density of a distribution:

In [ ]:
let

    # initialize -
    x = range(0.0, 1.0, length = 501); # rewards between 0 and 1

    # plot the reward distributions of arm 2 and arm 6 -
    plot(x, pdf.(arms[2], x), lw = 2, label = "Arm 2: (α, β) = (10, 2)", framestyle = :box,
        xlabel = "Reward r (AU)", ylabel = "Probability density (AU)", legend = :topleft)
    plot!(x, pdf.(arms[6], x), lw = 2, label = "Arm 6: (α, β) = (20, 10)")
end

__What do we see?__ The rewards of arm 2 cluster around its mean of 0.833 and those of arm 6 around 0.667, but the two distributions overlap. A single pull of arm 6 beats a single pull of arm 2 about one time in nine, so an algorithm must pull each arm several times before it can tell which one is better.

### Constants
Finally, let's set some constants we'll use in the subsequent tasks. See the comment beside the value for a description of what it is, its permissible values, etc.

In [ ]:
K = 7; # number of arms for the bandit
T = 5000; # number of rounds for each decision task
seed = 5800; # seed for the random number generator, so each game can be repeated

___

## Task 1: Run the Three Bandit Algorithms
In this task, we run the explore-first, $\epsilon$-greedy, and UCB1 algorithms on our `K`-arm bandit problem and see which arm each one pulls. We've created a custom type for each algorithm, which holds the number of arms `K`. We create an instance of the type using [the `build(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.build), then pass it, the number of rounds `T`, and our `world(...)` function to [the `solve(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/bandit/#VLDataScienceMachineLearningPackage.solve-Tuple{AbstractBanditAlgorithmModel}), which plays the game.

__Output__: With the `return_history = true` option, the call returns a `NamedTuple` with two fields: `actions`, the arm $a_t$ pulled in each round, and `rewards`, a $T\times K$ results matrix with one row per round and one column per arm. Row $t$ holds the reward $r_t$ in column $a_t$ and zeros everywhere else.

The algorithms and our `world(...)` function draw random numbers, so each game cell first sets the seed with [the `Random.seed!(...)` function](https://docs.julialang.org/en/v1/stdlib/Random/#Random.seed!) and gives the same game every time we run it.

__Prediction:__ After 5000 rounds, which algorithm will have pulled the best arm, arm 2, most often: {explore-first, $\epsilon$-greedy, or UCB1}?

### Explore-First Algorithm
The explore-first algorithm pulls each arm $N$ times, then pulls the arm with the highest sample mean for the remaining rounds. The package sets $N$ with the exploration count from the lecture:
$$
N=\min\left\{\left\lfloor T/K\right\rfloor,\;\max\left\{1,\;\operatorname{round}\left[(T/K)^{2/3}(\log T)^{1/3}\right]\right\}\right\}.
$$
The $\max$ makes sure every arm is pulled at least once, and the $\min$ makes sure the exploration fits in $T$ rounds. Neither limit is active for our $T$ and $K$. Let's compute $N$ to see how long the exploration phase lasts:


In [ ]:
N = round(Int, (T/K)^(2/3)*cbrt(log(T))); # pulls per arm while exploring, rounded to an integer
N = clamp(N, 1, fld(T, K)); # at least 1, and at most ⌊T/K⌋ so exploring fits in T rounds
println("each arm is pulled N = ", N, " times, so exploring takes the first ", K*N, " rounds")

We create [an instance of the `MyExploreFirstAlgorithmModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/bandit/#VLDataScienceMachineLearningPackage.MyExploreFirstAlgorithmModel) and play the game. We keep the returned `NamedTuple` in `game_explore_first` and its reward matrix in the `results_explore_first::Array{Float64,2}` variable:

In [ ]:
game_explore_first = let

    # initialize -
    Random.seed!(seed); # set the seed, so this cell gives the same game every time
    m = build(MyExploreFirstAlgorithmModel, (
        K = K, # number of arms
    ));

    # play the game -
    game = solve(m, T = T, world = world, return_history = true); # actions and rewards

    game # return
end;
results_explore_first = game_explore_first.rewards; # the T × K reward matrix, for the regret in Task 2

### Epsilon-Greedy Algorithm
The epsilon-greedy algorithm first pulls every arm once. After that initialization, it pulls a random arm with probability $\epsilon_t$, and otherwise pulls the arm with the highest sample mean. The package uses the capped exploration probability from the lecture:
$$
\epsilon_t=\min\left\{1,\left(\frac{K\log t}{t}\right)^{1/3}\right\}.
$$
For our $K=7$ arms, $\epsilon_t=1$ through round 21, so rounds 8 to 21 are all random pulls. After that, $\epsilon_t$ shrinks as the game goes on.

We create [an instance of the `MyEpsilonGreedyAlgorithmModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/bandit/#VLDataScienceMachineLearningPackage.MyEpsilonGreedyAlgorithmModel) and play the game. We keep the returned `NamedTuple` in `game_eps_greedy` and its reward matrix in the `results_eps_greedy::Array{Float64,2}` variable:


In [ ]:
game_eps_greedy = let

    # initialize -
    Random.seed!(seed); # set the seed, so this cell gives the same game every time
    m = build(MyEpsilonGreedyAlgorithmModel, (
        K = K, # number of arms
    ));

    # play the game -
    game = solve(m, T = T, world = world, return_history = true); # actions and rewards

    game # return
end;
results_eps_greedy = game_eps_greedy.rewards; # the T × K reward matrix, for the regret in Task 2

### UCB1 Algorithm
The UCB1 algorithm pulls each arm once. Then, in each round $t$, it pulls the arm with the highest upper confidence bound, the sample mean reward $\bar{\mu}(a)$ plus a confidence radius:
$$
\bar{\mu}(a)+\sqrt{\frac{2\log(t)}{N_{a}}},
$$
where $N_{a}$ is the number of times arm $a$ has been pulled so far. The radius is large for an arm we have pulled only a few times, so the algorithm is optimistic about the arms it knows least about.

We create [an instance of the `MyUCB1AlgorithmModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/bandit/#VLDataScienceMachineLearningPackage.MyUCB1AlgorithmModel) and play the game. We keep the returned `NamedTuple` in `game_UCB1` and its reward matrix in the `results_UCB1::Array{Float64,2}` variable:

In [ ]:
game_UCB1 = let

    # initialize -
    Random.seed!(seed); # set the seed, so this cell gives the same game every time
    m = build(MyUCB1AlgorithmModel, (
        K = K, # number of arms
    ));

    # play the game -
    game = solve(m, T = T, world = world, return_history = true); # actions and rewards

    game # return
end;
results_UCB1 = game_UCB1.rewards; # the T × K reward matrix, for the regret in Task 2

Let's look at the first ten rounds of the UCB1 game:

In [ ]:
results_UCB1[1:10, :] # rounds 1 to 10, all arms

__What do we see?__ Each row has one nonzero entry, the reward for the arm pulled in that round. In the first seven rounds, UCB1 pulls each arm once.

### Compare the Pull Counts
Let's count how many times each algorithm pulled each arm, using the `actions` field of each game:

In [ ]:
let

    # initialize -
    df = DataFrame(); # one row per arm

    # count the rounds in which each algorithm pulled each arm -
    for a ∈ 1:K
        push!(df, (
            arm = a,
            mean_reward = round(mean(arms[a]), digits = 3), # known only to us, the oracle
            explore_first = count(==(a), game_explore_first.actions), # ==(a) is a function that tests whether its input equals a
            eps_greedy = count(==(a), game_eps_greedy.actions),
            UCB1 = count(==(a), game_UCB1.actions),
        ));
    end

    # show the table -
    pretty_table(df;
        show_first_column_label_only = true,
        display_size = (-1, -1), # show every row and column
        alignment = :r # right-align every column
    );
end

__What do we see?__ With our seed, all three algorithms pulled arm 2 most often. Explore-first pulled every arm 163 times, then arm 2 for the rest of the game, 4,022 pulls of arm 2 in all. The $\epsilon$-greedy algorithm pulled arm 2 in 3,655 rounds and spread its other pulls about evenly over the other six arms. UCB1 pulled arm 2 in only 3,518 rounds and arm 5 in 946. Your counts will differ if you change the seed.

### Things to think about

__Question:__ Explore-first and $\epsilon$-greedy pulled the six other arms about equally often, while UCB1 pulled arm 5 almost a thousand times and arm 3 only 32 times. Why does UCB1 keep pulling arm 5? Why does it pull arm 3 so rarely? Briefly explain.

___


## Task 2: Compute the Empirical Regret for Each Algorithm
In this task, we compute the regret of each algorithm in our seeded game, then repeat the game with 30 seeds and four horizons to see which differences hold up on average. Finally, we compare how fast the average regret grows with the theoretical regret bounds.

__Regret__: Let $\mu^{\star}$ be the mean of the best arm, i.e., $\mu^{\star} = \max_{a\in\mathcal{A}}\mu(a)$. The regret $R(t)$ of an algorithm after $t$ rounds is defined as:
$$
\begin{align*}
R(t) = t\cdot\mu^{\star} - \sum_{s=1}^{t}r_{s}.
\end{align*}
$$
The first term is the expected reward from always choosing the best arm over the $t$ rounds. The second term is the total reward obtained by the agent over the $t$ rounds, where $r_{s}$ is the reward received at round $s$ for the action chosen by the agent.

We are the oracle, so we know the mean reward of every arm: $\mu^{\star}=10/12\approx 0.833$, the mean of arm 2. For each algorithm, we compute $R(t)$ after every round $t=1,2,\dots,T$ and store the curves in the `regret_curves::Dict{String, Array{Float64,1}}` dictionary:

In [ ]:
regret_curves = let

    # initialize -
    μstar = maximum(mean.(arms)); # μ⋆, the mean reward of the best arm: known only to us, the oracle
    games = [("Explore-first", results_explore_first), ("ε-greedy", results_eps_greedy), ("UCB1", results_UCB1)]; # (name, results array)
    regret_curves = Dict{String, Array{Float64,1}}(); # the regret after each round, for each algorithm

    # main loop -
    for (name, U) ∈ games
        r = vec(sum(U, dims = 2)); # the reward in each round: each row of U has at most one nonzero entry
        R = zeros(T); # the regret after each round
        total = 0.0; # the reward collected so far
        for t ∈ 1:T
            total = total + r[t]; # r₁ + r₂ + ... + rₜ
            R[t] = t*μstar - total; # R(t) = tμ⋆ - (r₁ + r₂ + ... + rₜ)
        end
        regret_curves[name] = R; # store the curve for this algorithm
    end

    regret_curves # return
end;

Let's plot the regret curves.

__Prediction:__ Which algorithm will have the smallest regret after 5000 rounds: {explore-first, $\epsilon$-greedy, or UCB1}?

In [ ]:
let

    # initialize -
    labels = ["Explore-first", "ε-greedy", "UCB1"]; # plot the curves in this order
    p = plot(framestyle = :box, xlabel = "Round t", ylabel = "Regret R(t) (AU)", legend = :topleft);

    # plot the regret after each round -
    for name ∈ labels
        plot!(p, 1:T, regret_curves[name], lw = 2, label = name);
    end

    p # return
end

__What do we see?__ Explore-first's regret climbs for the first 1,141 rounds, while it pulls every arm in turn, then levels off once it pulls only arm 2. The $\epsilon$-greedy and UCB1 curves separate within the first thousand rounds: $\epsilon$-greedy keeps pulling random arms, so its regret keeps climbing, while UCB1's grows more and more slowly.

Because $\epsilon$-greedy never stops exploring, its curve passes explore-first's at about round 3,300. After 5000 rounds, UCB1 has the smallest regret, about 232, compared with about 400 for explore-first and about 526 for $\epsilon$-greedy. These curves come from one game, and the random rewards make each curve wiggle up and down, even while an algorithm pulls the best arm.

### Repeat the Experiment Across Seeds and Horizons
One game is one draw of the random rewards and of each algorithm's random choices. To estimate the expected regret, we repeat each game with 30 seeds and average. Because we know the true arm means, we can also average the cumulative gap loss from the lecture, where $a_s$ is the arm selected at round $s$:
$$
G(t)=\sum_{s=1}^{t}\left(\mu^\star-\mu(a_s)\right).
$$
The gap loss charges each pull the mean reward given up by its arm, so it leaves out the luck of the individual rewards. Since $\mathbb{E}[R(t)]=\mathbb{E}[G(t)]$, the two averages should agree.

[The `compare_bandits(...)` function](docs/compare_bandits.md) plays all three algorithms at four horizons, $T$ = 1000, 2500, 5000, and 10,000, with 30 seeds each. Each horizon starts a new game, because explore-first sets its exploration count from $T$. We store the results in the `ensemble::NamedTuple`, which holds the gap loss and realized regret of every game, along with their averages and spreads.

__Prediction:__ Will UCB1, the algorithm with the smallest regret in our seeded game, also have the smallest average gap loss at every horizon?


In [ ]:
ensemble = let

    # initialize -
    horizons = [1000, 2500, 5000, 10_000]; # number of rounds in each fresh game
    seeds = collect(seed:(seed + 29)); # 30 seeds: 5800, 5801, ..., 5829

    # play 3 algorithms × 4 horizons × 30 seeds = 360 games -
    compare_bandits(arms, horizons, seeds)
end;


Let's look at the longest horizon, $T$ = 10,000. For each algorithm, the table shows the average gap loss over the 30 games, its standard deviation (how much the gap loss varies from game to game), its standard error (the standard deviation divided by $\sqrt{30}$, which measures how precisely the 30 games pin down the average), and the average realized regret:


In [ ]:
let

    # initialize -
    h = length(ensemble.horizons); # row of the longest horizon, T = 10,000
    df = DataFrame(); # one row per algorithm

    # averages and spreads over the 30 games -
    for j ∈ eachindex(ensemble.labels)
        push!(df, (
            algorithm = ensemble.labels[j],
            mean_gap = round(ensemble.mean_gap[h, j], digits = 1), # average gap loss G(T)
            sd_gap = round(ensemble.sd_gap[h, j], digits = 1), # how much G(T) varies between games
            se_gap = round(ensemble.se_gap[h, j], digits = 1), # uncertainty in the average
            mean_realized = round(ensemble.mean_realized[h, j], digits = 1), # average realized regret R(T)
        ));
    end

    # show the table -
    pretty_table(df;
        show_first_column_label_only = true,
        display_size = (-1, -1), # show every row and column
        alignment = [:l, :r, :r, :r, :r] # left-align the names and right-align the numbers
    );
end

__What do we see?__ UCB1 has the smallest average gap loss, about 286, then explore-first, about 621, and $\epsilon$-greedy, about 873. In each row, the average gap loss and the average realized regret agree to within a few units, consistent with $\mathbb{E}[R]=\mathbb{E}[G]$.

Explore-first's gap loss has a standard deviation of zero across these 30 games: it committed to arm 2 in all of them, so its gap loss is always the cost of its exploration phase. Its realized regret still varies, because the rewards do.

Let's plot the average gap loss against the horizon. Each ribbon extends two standard errors above and below the average:


In [ ]:
let

    # initialize -
    p = plot(framestyle = :box, xlabel = "Horizon T (rounds)",
        ylabel = "Average gap loss G(T) (AU)", legend = :topleft); # empty plot, then one line per algorithm

    # plot the average gap loss at each horizon, with a band of ± 2 standard errors -
    for j ∈ eachindex(ensemble.labels)
        plot!(p, ensemble.horizons, ensemble.mean_gap[:, j],
            ribbon = 2 .* ensemble.se_gap[:, j], fillalpha = 0.15, # shaded band around the average
            lw = 2, marker = :circle, label = ensemble.labels[j]);
    end

    p # return
end

__What do we see?__ With our arms and seeds, UCB1 has the smallest average gap loss at every horizon; in fact, it had a smaller gap loss than both other algorithms in each of the 30 games at every horizon. The standard errors are at most about 3.1, so the ribbons are too thin to see: the differences between the algorithms are far larger than the uncertainty in the averages.

### Theoretical Regret Bounds
Finally, let's compare the growth of the average gap loss with the theoretical regret bounds from the lecture:

* __Explore-first__ and __$\epsilon$-greedy__: The expected regret over $T$ rounds is at most a constant times ${T}^{2/3}\cdot\left(K\cdot\log{T}\right)^{1/3}$, which grows like $T^{2/3}$ apart from the logarithm.
* __UCB1__: The expected regret over $T$ rounds is at most a constant times $\sqrt{K\cdot{T}\cdot\log{T}}$, which grows like $\sqrt{T}$ apart from the logarithm, the slowest growth of the three.

The bounds hide unknown constants in the $\mathcal{O}(\cdot)$ notation, so they do not tell us the size of the regret. However, when we divide a bound at one horizon by the same bound at another horizon, the constant cancels. We compare the two longest horizons, $T_{1}=5000$ and $T_{2}=10{,}000$. For each algorithm, the table shows the average gap loss at both horizons, how much it grew, and how much the bound grew.

__Prediction:__ When we double the horizon, will the average gap loss of each algorithm grow {faster than, more slowly than, or about as fast as} its bound?

In [ ]:
let

    # initialize -
    df = DataFrame(); # one row per algorithm
    T₁ = ensemble.horizons[end - 1]; # the second-longest horizon, T₁ = 5000
    T₂ = ensemble.horizons[end]; # the longest horizon, T₂ = 10,000

    # how much each bound grows from T₁ to T₂; the unknown constant cancels in the ratio -
    growth_explore = ((T₂^(2/3))*(K*log(T₂))^(1/3))/((T₁^(2/3))*(K*log(T₁))^(1/3)); # explore-first and ε-greedy bound
    growth_UCB1 = sqrt(K*T₂*log(T₂))/sqrt(K*T₁*log(T₁)); # UCB1 bound
    bound_growth = [growth_explore, growth_explore, growth_UCB1]; # in the same order as ensemble.labels

    # compare how much the average gap loss grew with how much the bound grew -
    for j ∈ eachindex(ensemble.labels)
        G₁ = ensemble.mean_gap[end - 1, j]; # average gap loss G(T₁)
        G₂ = ensemble.mean_gap[end, j]; # average gap loss G(T₂)
        push!(df, (
            algorithm = ensemble.labels[j],
            mean_gap_T₁ = round(G₁, digits = 1),
            mean_gap_T₂ = round(G₂, digits = 1),
            gap_growth = round(G₂/G₁, digits = 2), # G(T₂)/G(T₁)
            bound_growth = round(bound_growth[j], digits = 2), # the bound at T₂ divided by the bound at T₁
        ));
    end

    # show the table -
    pretty_table(df;
        show_first_column_label_only = true,
        display_size = (-1, -1), # show every row and column
        alignment = [:l, :r, :r, :r, :r] # left-align the names and right-align the numbers
    );
end

__What do we see?__ Explore-first committed to arm 2 in every game, so its gap loss is the cost of its exploration phase. Its exploration count $N$ grows from 163 to 266 pulls per arm, a factor of about 1.63, which matches its bound, because $N$ and the bound grow with $T$ in the same way.

The $\epsilon$-greedy gap loss grows by about 1.64, close to its bound's 1.63. UCB1's gap loss grows by only about 1.30, more slowly than its bound's 1.47.

### Things to think about

__Question:__ In our seeded game, UCB1 pulled the best arm less often than $\epsilon$-greedy (3,518 times versus 3,655), yet it has the smallest regret. How can that be? Why did its gap loss grow the least when we doubled the number of rounds? Briefly explain.

___


## Task 3: Implement Thompson Sampling
In this task, we implement Thompson sampling and use it to solve a binary Bernoulli bandit problem, where every reward is a success, $r=1$, or a failure, $r=0$.

Suppose each arm $a\in\mathcal{A}$ succeeds with a probability $\theta_{a}$ that the agent _does not know_. However, we are [the Oracle](https://en.wikipedia.org/wiki/The_Oracle_(The_Matrix)), so we know the actual probabilities. By experimenting in the world, can the agent learn which arm is most likely to succeed? Let's set up a world with three arms, with success probabilities $\theta_{1}=0.90$, $\theta_{2}=0.60$, and $\theta_{3}=0.80$:

In [ ]:
θ = [0.90, 0.60, 0.80]; # the probability of success of each arm: only the world knows these

### Complete the Thompson Sampling Function
The agent models its uncertainty about each $\theta_{a}$ with a Beta distribution. Before any pulls, its belief about arm $a$, the __prior__, is $\text{Beta}(\alpha(a),\beta(a))$; we use $\alpha(a)=\beta(a)=1$, which is uniform on $[0,1]$. After $S(a)$ successes and $F(a)$ failures on arm $a$, its updated belief, the __posterior__, is $\text{Beta}(\alpha(a)+S(a),\beta(a)+F(a))$.

Let's write Thompson sampling ourselves, following the lecture's algorithm. Open [`src/Compute.jl`](src/Compute.jl) and complete [the `my_thompson_sampling(...)` function](src/Compute.jl). Everything except four lines is already written, and a TODO comment marks each step:

> __Implementation tasks__
>
> * __TODO 1:__ Sample from the posterior of each arm. Replace `0.0` in `p[a] = 0.0` with one sample from $\text{Beta}(\alpha(a)+S(a),\beta(a)+F(a))$, drawn with the random number generator `rng`.
> * __TODO 2:__ Choose the arm with the highest sample. Replace `1` in `aₜ = 1` with the arm $a$ that maximizes the sample $p(a)$.
> * __TODO 3:__ Update the counts of the arm we pulled. Replace the right-hand sides of the two lines so that $S(a_{t})\gets S(a_{t})+r_{t}$ and $F(a_{t})\gets F(a_{t})+(1-r_{t})$.

We play $T=5000$ rounds against [the `bernoulli_world(...)` function](docs/compare_bandits.md#bernoulli-world), which returns a reward of `1` with probability $\theta_{a}$ and `0` otherwise, using [a Bernoulli distribution](https://en.wikipedia.org/wiki/Bernoulli_distribution). The agent and the world share one random number generator, created from our `seed` by [the `MersenneTwister(...)` function](https://docs.julialang.org/en/v1/stdlib/Random/#Random.MersenneTwister).

We save the returned `NamedTuple` in the `result` variable. Its fields `actions` and `rewards` hold the arm pulled and the reward received in each round, and `S` and `F` hold the successes and failures on each arm.

Save [`src/Compute.jl`](src/Compute.jl) and re-run the setup cell at the top of the notebook to reload your edits, then run the cell below. The untouched function throws an error, because it never updates the success and failure counts:

In [ ]:
result = let

    # initialize -
    rng = MersenneTwister(seed); # one random number generator for the agent and the world
    world_function = a -> bernoulli_world(a, θ, rng); # the agent sees only the reward of the arm it pulls

    # play T rounds with Thompson sampling -
    L10dThompson.my_thompson_sampling(world_function, length(θ), T, rng = rng) # the untouched stub throws an error
end;
println("pulls of each arm: ", result.S .+ result.F)

### Check: Your Thompson Sampling

Every round adds one success or one failure to the arm we pulled, so the counts must agree with the actions and the rewards, and the same seed must give the same game. We also play a test bandit whose best arm is the last one, $\theta=(0.70, 0.50, 0.90)$, which Thompson sampling should pull in most rounds:

In [ ]:
@testset "My Thompson sampling" begin

    # every round adds one success or one failure to the arm we pulled -
    @test sum(result.S) + sum(result.F) == T
    @test result.S .+ result.F == [count(result.actions .== a) for a ∈ eachindex(θ)] # pulls of each arm
    @test result.S == [sum(result.rewards[result.actions .== a]) for a ∈ eachindex(θ)] # successes on each arm

    # the same seed gives the same game, every time we play it -
    for k ∈ 1:2
        rng = MersenneTwister(seed); # a new random number generator with the same seed
        again = L10dThompson.my_thompson_sampling(a -> bernoulli_world(a, θ, rng), length(θ), T, rng = rng);
        @test again.actions == result.actions
    end

    # a test bandit whose best arm is the last one -
    θ_test = [0.70, 0.50, 0.90]; # success probabilities of the test bandit
    rng_test = MersenneTwister(1234); # its own random number generator
    test = L10dThompson.my_thompson_sampling(a -> bernoulli_world(a, θ_test, rng_test), 3, 2000, rng = rng_test);
    @test count(test.actions .== 3) > 1000 # the best arm is pulled in most of the 2000 rounds
    @test abs((1 + test.S[3])/(2 + test.S[3] + test.F[3]) - 0.90) < 0.05 # its posterior mean is close to 0.9
end;

### What Did the Agent Learn?
Let's compare what the agent believes with the ground truth that only we know. The agent's estimate of $\theta_{a}$ is the posterior mean:
$$
\hat{\theta}_{a} = \frac{\alpha(a)+S(a)}{\alpha(a)+\beta(a)+S(a)+F(a)}.
$$

__Prediction:__ For which arm will the estimate $\hat{\theta}_{a}$ be closest to the true value: {arm 1, arm 2, or arm 3}?

In [ ]:
let

    # initialize -
    df = DataFrame(); # one row per arm
    α = 1.0; # prior parameter α(a), the same for every arm
    β = 1.0; # prior parameter β(a), the same for every arm

    # compare the posterior mean with the true success probability -
    for a ∈ eachindex(θ)
        S = result.S[a]; # successes on arm a
        F = result.F[a]; # failures on arm a
        push!(df, (arm = a, θ = θ[a], pulls = S + F, S = S, F = F, θ̂ = round((α + S)/(α + β + S + F), digits = 3)));
    end

    # show the table -
    pretty_table(df;
        show_first_column_label_only = true,
        display_size = (-1, -1), # show every row and column
        alignment = :r # right-align every column
    );
end

__What do we see?__ With our seed, the agent pulled arm 1 in 4,908 of the 5000 rounds, and its estimate of $\theta_{1}$, 0.905, is close to the true 0.90. It pulled arm 3 only 81 times and arm 2 only 11 times, so its estimates for those arms, 0.819 and 0.538, are based on far fewer observations.

Let's plot the posterior of each arm with [the `pdf(...)` function](https://juliastats.org/Distributions.jl/stable/univariate/#Distributions.pdf-Tuple{UnivariateDistribution,%20Real}):

In [ ]:
let

    # initialize -
    x = range(0.0, 1.0, length = 2001); # success probabilities between 0 and 1, on a fine grid: the posterior of arm 1 is narrow
    p = plot(framestyle = :box, xlabel = "Probability of success (AU)", ylabel = "Probability density (AU)", legend = :topleft);

    # plot the posterior of each arm -
    for a ∈ eachindex(θ)
        d = Beta(1.0 + result.S[a], 1.0 + result.F[a]); # the posterior of arm a, with α(a) = β(a) = 1
        plot!(p, x, pdf.(d, x), lw = 2, label = "Arm $(a): θ = $(θ[a])");
    end

    p # return
end

__What do we see?__ The posterior of arm 1 is a tall, narrow peak near 0.9: after thousands of pulls, the agent is nearly sure of $\theta_{1}$. The posterior of arm 3 is wider, and the posterior of arm 2 is broad.

__Prediction:__ Suppose arm 3 is almost as good as arm 1, with $\theta_{3}=0.88$ instead of 0.80. Will Thompson sampling pull arm 3 {more often, less often, or about as often} as before?

In [ ]:
let

    # initialize -
    θ_close = [0.90, 0.60, 0.88]; # arm 3 is now almost as good as arm 1
    rng = MersenneTwister(seed); # the same seed as before
    world_function = a -> bernoulli_world(a, θ_close, rng); # the agent sees only the reward of the arm it pulls

    # play T rounds with Thompson sampling -
    close_result = L10dThompson.my_thompson_sampling(world_function, length(θ_close), T, rng = rng);
    println("θ = ", θ_close, ": pulls of each arm ", close_result.S .+ close_result.F)
    println("posterior means: ", round.((1 .+ close_result.S) ./ (2 .+ close_result.S .+ close_result.F), digits = 3))
end

__What do we see?__ With $\theta_{3}=0.88$, the agent pulled arm 3 in 551 rounds instead of 81. It takes more pulls to tell two nearly equal arms apart.

### Things to think about

__Question:__ After 5000 rounds, the agent's estimate of $\theta_{2}$ is still rough, because it pulled arm 2 only 11 times. Is that a problem for the agent? Why does Thompson sampling pull an arm so rarely, even before it knows that arm's success probability well? How is this like what UCB1 did with arm 3 in Task 1? Briefly explain.

___

## Summary
In this lab, we ran the explore-first, $\epsilon$-greedy, and UCB1 algorithms on a seven-arm bandit, measured their regret in one game and averaged over many, compared how fast the average regret grew with the theoretical bounds, and implemented Thompson sampling for a binary Bernoulli bandit.

> __Key Takeaways:__
>
> * __Three ways to balance exploration and exploitation:__ All three algorithms found the best arm, but they explored in different ways. Explore-first and epsilon-greedy spread their exploration evenly over the arms, while UCB1 spent most of its exploration on the arms whose mean rewards were close to the best.
>
> * __Regret depends on which arms we explore:__ UCB1 pulled the best arm less often than epsilon-greedy but had the smallest regret, because most of its other pulls went to arms that were almost as good. Averaged over 30 games, UCB1 had the smallest regret at all four horizons we tried, and its regret grew the most slowly, which agrees with its slower-growing regret bound.
>
> * __Thompson sampling keeps a probability model of each arm:__ We completed a Thompson sampling function that samples from the posterior of each arm and updates the success and failure counts after each pull. It pulled the best arm in almost every round, and it pulled the worse arms only rarely, even before it knew their success probabilities well.

We can now change the arms, the number of rounds, or the seed and see how the pulls and the regret of each algorithm respond.

___